# Week 10: TF-IDF, BM25, and ranking metrics (capstone milestone M2)

You'll build a real ranking function, BM25, which is still the baseline that every neural retriever has to beat. Then you'll measure it properly.

| Session | Topic | Time |
|---|---|---|
| Day 1 | TF-IDF and cosine ranking over your inverted index | 2.5 h |
| Day 2 | BM25: derive the intuition, implement it, test it | 2.5 h |
| Day 3 | Ranking metrics from scratch: P@k, recall@k, MAP, nDCG | 2.5 h |
| Day 4 | Evaluate on NFCorpus; tune on **dev**, report on **test** | 2 h |
| Review | Spaced recall | 0.5 h |

**Read:** *IIR*, chapter 6 (sections 6.2–6.4), chapter 8 (sections 8.1–8.4), and section 11.4.3 (Okapi BM25).

### Milestone M2
- [ ] `medlab/ranking.py` (TF-IDF and BM25) and `medlab/metrics.py`, each with tests including hand-computed cases.
- [ ] BM25 matches the brute-force reference in the checker exactly.
- [ ] A results table for TF-IDF and BM25 (default and tuned) on the **test** split, with `k1` and `b` tuned on **dev** only.

In [ ]:
import sys, importlib, subprocess
from pathlib import Path
for p in ["..", "../capstone"]:
    if str(Path(p).resolve()) not in sys.path:
        sys.path.insert(0, str(Path(p).resolve()))
CAP = Path("../capstone")

def run_tests(*paths):
    """Run pytest inside the capstone folder and return its exit code (0 = all passed)."""
    r = subprocess.run([sys.executable, "-m", "pytest", "-q", *paths], cwd=CAP, capture_output=True, text=True)
    print(r.stdout[-2500:], r.stderr[-800:])
    return r.returncode

def fresh(module_name):
    """Import (or re-import after you edited it) a module from your capstone package."""
    importlib.invalidate_caches()          # notice files you created after the kernel started
    return importlib.reload(importlib.import_module(module_name))

In [ ]:
import math
import numpy as np
from course.data import load_nfcorpus
index_mod, text_mod = fresh("medlab.index"), fresh("medlab.text")
pp = text_mod.preprocess
corpus, queries, qrels = load_nfcorpus("test")
doc_texts = {d: v["title"] + " " + v["text"] for d, v in corpus.items()}
index = index_mod.InvertedIndex.build(doc_texts, pp)
print(index.n_docs, "docs,", len(queries), "test queries")

---
# Day 1: TF-IDF (2.5 h)

Week 4's raw-count ranking had two problems: **common words** ("study", "patients") dominate, and **long documents** win just by being long.

- **Term frequency (TF):** use $1 + \log(tf)$ rather than raw $tf$. Ten mentions aren't 10× as relevant as one.
- **Inverse document frequency (IDF):** $\text{idf}(t) = \log \frac{N}{df(t)}$. Rare terms are informative; a term found in every document gets weight 0.
- **Cosine normalisation:** divide by the document vector's length, so long documents don't win automatically (Week 5!).

$$\text{score}(q, d) = \frac{\sum_{t \in q} w_{t,q}\, w_{t,d}}{\|w_d\|}, \quad w_{t,d} = (1 + \log tf_{t,d}) \cdot \text{idf}(t)$$
(Dividing by $\|w_q\|$ doesn't change the ranking for a fixed query, so we skip it.)

### Exercise 10.1: `TfIdf` in `capstone/medlab/ranking.py`
```python
class TfIdf:
    def __init__(self, index)              # precompute idf for every term and ||w_d|| for every doc
    def score(self, q_tokens) -> dict[str, float]   # only documents containing ≥1 query term
    def search(self, q_tokens, k=100) -> list[tuple[str, float]]  # sorted by score desc, then doc_id asc
```
**Efficiency rule:** loop over the query terms' **postings**, never over all documents. That's the whole point of an inverted index.

In [ ]:
# Implement TfIdf in VS Code (and tests in capstone/tests/test_ranking.py), then run the checker.

In [ ]:
ranking = fresh("medlab.ranking")
tfidf = ranking.TfIdf(index)
q_id, q_text = next(iter(queries.items()))
res = tfidf.search(pp(q_text), k=10)
print(q_text, "→", res[:3])
# brute-force reference over ALL documents
def ref_tfidf(q_tokens, d):
    N = index.n_docs
    w = lambda t, tf: (1 + math.log(tf)) * math.log(N / index.df(t))
    norm = math.sqrt(sum(w(t, p[d]) ** 2 for t, p in index.postings.items() if d in p))
    qtf = {t: q_tokens.count(t) for t in set(q_tokens) if t in index.postings}
    return sum((1 + math.log(c)) * math.log(N / index.df(t)) * w(t, index.postings[t][d]) for t, c in qtf.items() if d in index.postings[t]) / norm
for d, s in res[:5]:
    assert abs(s - ref_tfidf(pp(q_text), d)) < 1e-9
assert res == sorted(res, key=lambda x: (-x[1], x[0]))
print("✅ 10.1 passed")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

In `__init__`: `self.idf = {t: math.log(N / len(p)) for t, p in index.postings.items()}`. Compute the doc norms with one pass over all postings, accumulating `w**2` per document in a `Counter`.

</details>

<details><summary>Hint 2</summary>

In `score`: `Counter(q_tokens)` gives the query tf. For each query term in the index, loop over `index.postings[t].items()` and add `w_tq * w_td` to `scores[d]`. Divide by the norm at the end.

</details>


---
# Day 2: BM25 (2.5 h)

BM25 comes from a probabilistic model of relevance, but you can understand every piece intuitively:
$$\text{BM25}(q, d) = \sum_{t \in q} \underbrace{\log\left(\frac{N - df_t + 0.5}{df_t + 0.5} + 1\right)}_{\text{idf: rare terms matter more}} \cdot \underbrace{\frac{tf_{t,d}\,(k_1 + 1)}{tf_{t,d} + k_1\left(1 - b + b\,\frac{|d|}{\text{avgdl}}\right)}}_{\text{saturating TF with length normalisation}}$$

- **$k_1$** (typically 0.9–2.0) controls **saturation**: the score rises with $tf$ but levels off at $k_1 + 1$. The 10th mention adds little.
- **$b$** (0–1, typically 0.75) controls **length normalisation**: $b = 1$ fully penalises long documents; $b = 0$ ignores length.
- The "+1" inside the log keeps idf positive even for very common terms (this is the Lucene variant).

In [ ]:
import matplotlib.pyplot as plt
tf = np.arange(0, 21)
for k1 in [0.5, 1.2, 3.0]:
    plt.plot(tf, tf * (k1 + 1) / (tf + k1), label=f"k1={k1}")
plt.plot(tf, tf / 4, "k--", label="raw tf (scaled)"); plt.xlabel("term frequency"); plt.ylabel("TF component (|d| = avgdl)")
plt.legend(); plt.title("BM25 term-frequency saturation"); plt.show()

### Exercise 10.2: `BM25` in `ranking.py`
`BM25(index, k1=1.2, b=0.75)` with the same `score` / `search` interface as `TfIdf`. Add tests to `tests/test_ranking.py`, including one that `b=0` makes document length irrelevant.

In [ ]:
# Implement BM25, then run the checker.

In [ ]:
ranking = fresh("medlab.ranking")
assert run_tests("tests/test_ranking.py") == 0
bm25 = ranking.BM25(index, k1=1.2, b=0.75)
def ref_bm25(q_tokens, d, k1=1.2, b=0.75):
    N, s = index.n_docs, 0.0
    for t in q_tokens:
        if d in index.postings.get(t, {}):
            df_, tf_ = index.df(t), index.postings[t][d]
            idf = math.log((N - df_ + 0.5) / (df_ + 0.5) + 1)
            s += idf * tf_ * (k1 + 1) / (tf_ + k1 * (1 - b + b * index.doc_len[d] / index.avgdl))
    return s
for q_text in list(queries.values())[:5]:
    for d, s in bm25.search(pp(q_text), k=5):
        assert abs(s - ref_bm25(pp(q_text), d)) < 1e-9
print("✅ 10.2 passed")

---
# Day 3: Ranking metrics (2.5 h)

For one query with a ranked list and graded judgements `rel = {doc_id: grade}`:
- **P@k** = (relevant docs in the top k) / k
- **Recall@k** = (relevant docs in the top k) / (total relevant)
- **AP** (average precision) = the mean of P@i over the ranks *i* where a relevant document appears, divided over *all* relevant docs (missed ones contribute 0). **MAP** = the mean of AP over queries.
- **nDCG@k** uses the grades and rewards putting the *most* relevant documents first:
$$\text{DCG@k} = \sum_{i=1}^{k} \frac{\text{grade}_i}{\log_2(i + 1)}, \qquad \text{nDCG@k} = \frac{\text{DCG@k}}{\text{IDCG@k}}$$
where IDCG is the DCG of the *ideal* ordering. nDCG@10 is the main metric of the BEIR benchmark.

**Unjudged documents count as non-relevant.** That's a known bias, and it goes in your limitations section.

### Exercise 10.3: `capstone/medlab/metrics.py`
```python
precision_at_k(ranked, rel, k)    recall_at_k(ranked, rel, k)    average_precision(ranked, rel)
ndcg_at_k(ranked, rel, k)
per_query(run, qrels, k=10) -> {qid: {"ndcg@10":…, "P@10":…, "recall@100":…, "MAP":…}}
evaluate_run(run, qrels, k=10) -> {metric: mean over ALL queries in qrels}
```
A **run** is `{qid: {doc_id: score}}`. Sort by score descending, breaking ties by doc_id. A query missing from the run scores 0. **Write the tests first, computing at least two cases by hand on paper.**

In [ ]:
# Implement metrics.py and tests/test_metrics.py, then run the checker.

In [ ]:
metrics = fresh("medlab.metrics")
assert run_tests("tests/test_metrics.py") == 0
rel = {"a": 2, "c": 1, "z": 1}
ranked = ["a", "b", "c", "d"]
assert metrics.precision_at_k(ranked, rel, 4) == 0.5
assert abs(metrics.recall_at_k(ranked, rel, 4) - 2 / 3) < 1e-12
assert abs(metrics.average_precision(ranked, rel) - (1 + 2 / 3) / 3) < 1e-12
idcg = 2 + 1 / math.log2(3) + 1 / math.log2(4)
assert abs(metrics.ndcg_at_k(ranked, rel, 4) - (2 + 1 / math.log2(4)) / idcg) < 1e-12
assert metrics.evaluate_run({}, {"q": rel})["ndcg@10"] == 0.0
print("✅ 10.3 passed")

---
# Day 4: Evaluate, tune on dev, report on test (2 h)

**The protocol** (write it down *before* you look at the test numbers):
1. Tune `k1 ∈ {0.6, 0.9, 1.2, 1.5, 2.0}` and `b ∈ {0.3, 0.5, 0.75, 0.9}` on the **dev** queries.
2. Fix the best setting. Evaluate TF-IDF, BM25 with defaults, and BM25 tuned **once** on test.

In [ ]:
def make_run(ranker, queries, k=100):
    return {qid: dict(ranker.search(pp(q), k=k)) for qid, q in queries.items()}

_, dev_queries, dev_qrels = load_nfcorpus("dev")

### Exercise 10.4: Grid search on dev
Fill `grid` with `{(k1, b): dev nDCG@10}` and set `best = (k1, b)`.

In [ ]:
grid = {}
best = None

In [ ]:
assert len(grid) == 20 and best == max(grid, key=grid.get)
print("best on dev:", best, round(grid[best], 4))
import pandas as pd
rows = []
for name, ranker in [("TF-IDF", ranking.TfIdf(index)), ("BM25 (1.2, 0.75)", ranking.BM25(index)), (f"BM25 tuned {best}", ranking.BM25(index, *best))]:
    rows.append({"system": name, **metrics.evaluate_run(make_run(ranker, queries), qrels)})
results = pd.DataFrame(rows).set_index("system").round(4)
print(results)
Path("../capstone/results").mkdir(parents=True, exist_ok=True)
results.to_csv("../capstone/results/week10_lexical.csv")
print("✅ 10.4 passed (milestone M2)")

**Hints** (open one at a time, and note in your log which ones you needed):

<details><summary>Hint 1</summary>

Two nested loops over k1 and b: build `ranking.BM25(index, k1, b)`, make a run on `dev_queries`, and store `metrics.evaluate_run(run, dev_qrels)["ndcg@10"]`.

</details>


### Sanity check against a library
The BEIR paper reports BM25 nDCG@10 ≈ **0.325** on NFCorpus (with Elasticsearch, which uses a different tokeniser and stemmer). Yours should land in the same neighbourhood, roughly 0.30–0.34. If it's far off, suspect preprocessing first, then the metric code.

You can also compare with the `rank_bm25` library. It uses a slightly different idf (no +1, plus an epsilon floor), so expect *close*, not identical, numbers.

In [ ]:
from rank_bm25 import BM25Okapi
ids = list(doc_texts)
lib = BM25Okapi([pp(doc_texts[d]) for d in ids], k1=1.2, b=0.75)
lib_run = {}
for qid, q in queries.items():
    s = lib.get_scores(pp(q)); top = np.argsort(-s)[:100]
    lib_run[qid] = {ids[i]: float(s[i]) for i in top if s[i] > 0}
print("rank_bm25 nDCG@10:", round(metrics.evaluate_run(lib_run, qrels)["ndcg@10"], 4), "| yours:", results.iloc[1]["ndcg@10"])

✍️ *Did tuning on dev improve test? By how much? Is that difference meaningful? (You'll test it properly in Week 12.) Which queries does BM25 fail on completely (nDCG = 0)? Look at 3 of them, and the relevant docs. Why does it fail?* (double-click to edit)



---
# Review (30 min)
1. *(Week 5)* Why is cosine similarity length-independent?
2. *(Week 7)* What is the idf of a term that occurs in every document, and why is that sensible?
3. *(Week 8)* Why must `k1` and `b` be tuned on dev, not test?
4. What do `k1` and `b` each control? Sketch the saturation curve from memory.
5. Compute the nDCG@3 of the ranking [rel=0, rel=2, rel=1] by hand.
6. Why loop over postings rather than over documents?

Commit to the `medsearch-lab` repo: `Add TF-IDF and BM25 rankers with tests`, `Add ranking metrics`, `Add lexical baseline results`.

✍️ *Your answer:* (double-click to edit)

